# Experimentos v7 — sobre la base de verscla07

Arranca en segundos desde `resultados/base/` (guardado al final de `competencia_v5.ipynb`).
Cada experimento llama a `lab.evaluar(...)`, que lo deja en `resultados/registro.csv`.

**Regla de decisión:** para cambiar la entrega, la meseta (ganancia promedio entre 12.500 y 15.000 envíos en junio)
tiene que superar a verscla07 (**432,8**) por **más de 10 M** y ganar en casi todos los cortes.

In [1]:
import importlib, os, json, duckdb
import numpy as np, pandas as pd, lightgbm as lgb
import lab
importlib.reload(lab)   # si edito lab.py, vuelvo a correr esta celda

tr, te, fu, cfg, probs = lab.cargar()
feats = cfg['features_a']
params, n_iter = cfg['params'], cfg['best_iter']          # LightGBM de verscla05
params_v2, iter_v2 = cfg['params_v2'], cfg['iter_v2']      # LightGBM Optuna (verscla06)
px, semillas, ratio = cfg['px'], cfg['mis_semillas'], cfg['ratio']

# Referencias
ens3 = lab.ensamble(probs['prob_junio_v05'], probs['prob_junio_v06'], probs['prob_junio_xgb'])
lab.evaluar('ref verscla05 (lgbm amex)', probs['prob_junio_v05'], te)
lab.evaluar('ref verscla07 (ens3)', ens3, te)

abril (163284, 214) | junio (164114, 214) | agosto (164647, 214) | 203 variables | probabilidades: ['prob_agosto_v05', 'prob_agosto_v06', 'prob_agosto_xgb', 'prob_junio_v03', 'prob_junio_v05', 'prob_junio_v06', 'prob_junio_xgb']
ref verscla05 (lgbm amex)                  meseta  426.9 | vs ref   -5.9 | máx  431.2 en 13000
ref verscla07 (ens3)                       meseta  432.8 | vs ref   -0.0 | máx  436.7 en 13000


432.75833333333327

## Fase 1 — Verscla02 (el agente) y las 29 variables bancarias

Verscla02 ve el problema de otra forma: ~1.250 variables (lags 1-3, deltas, pendientes, variables bancarias)
y pocas por árbol (feature_fraction 0,11). Modelos con información distinta son los que mejor ensamblan.

In [2]:
ruta02 = os.path.expanduser('~/Desktop/agente_churn/data/exp/exp7001/')
feats02 = lgb.Booster(model_file=ruta02 + 'modelos/lgbm_semilla_128693.txt').feature_name()
cfg02 = json.load(open(ruta02 + 'optuna_mejores.json'))
params02 = {k: v for k, v in cfg02['params'].items() if k != 'metric'}
iter02 = cfg02['best_iter']

fe02 = duckdb.sql(f"select * from read_parquet('{ruta02}fe.parquet')").df()
fe02['clase_binaria2'] = fe02['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']).astype(int)
tr02 = fe02[fe02['foto_mes'] == 202104].reset_index(drop=True)
te02 = lab.mismo_orden(fe02[fe02['foto_mes'] == 202106], te)          # mismo orden de clientes que te
print(len(feats02), 'variables en verscla02 | junio alineado (tiene que dar 1.0):', (te02['clase_ternaria'].values == te['clase_ternaria'].values).mean())

1253 variables en verscla02 | junio alineado (tiene que dar 1.0): 1.0


In [3]:
# A) verscla02 reentrenado con abril (debería dar cerca de los 404 M del pipeline)
prob02 = lab.semillerio_lgb(tr02, te02, feats02, params02, iter02, semillas)
lab.evaluar('verscla02 (agente, abril->junio)', prob02, te)
lab.evaluar('ens2 verscla05 + verscla02', lab.ensamble(probs['prob_junio_v05'], prob02), te)
lab.evaluar('ens4 ens3 + verscla02', lab.ensamble(probs['prob_junio_v05'], probs['prob_junio_v06'],
                                                  probs['prob_junio_xgb'], prob02), te)
print(lab.gana_en_cortes(lab.ensamble(probs['prob_junio_v05'], probs['prob_junio_v06'],
                                      probs['prob_junio_xgb'], prob02), ens3, te))

verscla02 (agente, abril->junio)           meseta  380.9 | vs ref  -51.9 | máx  396.6 en 15500
ens2 verscla05 + verscla02                 meseta  419.6 | vs ref  -13.2 | máx  422.9 en 13500
ens4 ens3 + verscla02                      meseta  433.1 | vs ref   +0.3 | máx  437.8 en 14000
{'gana': 4, 'empata': 0, 'pierde': 7}


In [4]:
# B) las variables bancarias del agente sumadas a las 203 del notebook
banc = ['tc_saldo_total', 'tc_consumo_total', 'tc_limite_total', 'tc_pagado_total', 'tc_uso_limite',
        'tc_pago_sobre_saldo', 'tc_pagominimo_sobre_saldo', 'tc_peor_status', 'tc_delinquency', 'tc_activas',
        'sin_visa', 'inversiones_pesos', 'prestamos_total', 'ratio_tc_saldos', 'ratio_pasivos_activos',
        'cobra_sueldo', 'transferencias_neto', 'trx_digitales', 'trx_atencion', 'ratio_digital',
        'debitos_automaticos_total', 'pagos_servicios_total', 'ratio_ctrx_antiguedad', 'seguros_total',
        'rentabilidad_por_producto', 'comisiones_por_producto', 'edad_al_alta']
banc = [c for c in banc if c in fe02.columns and c not in tr.columns]   # sin duplicar las que ya están
cols_b = ['numero_de_cliente', 'foto_mes'] + banc
tr_b = tr.merge(fe02[cols_b], on=['numero_de_cliente', 'foto_mes'], how='left')
te_b = te.merge(fe02[cols_b], on=['numero_de_cliente', 'foto_mes'], how='left')
fu_b = fu.merge(fe02[cols_b], on=['numero_de_cliente', 'foto_mes'], how='left')
feats_b = feats + banc
print(len(banc), 'bancarias nuevas ->', len(feats_b), 'variables')

prob_b = lab.semillerio_lgb(tr_b, te_b, feats_b, params, n_iter, semillas)
lab.evaluar(f'lgbm amex + {len(banc)} bancarias', prob_b, te)
lab.evaluar('ens3 con bancarias', lab.ensamble(prob_b, probs['prob_junio_v06'], probs['prob_junio_xgb']), te)
lab.evaluar('ens4 con bancarias + verscla02',
            lab.ensamble(prob_b, probs['prob_junio_v06'], probs['prob_junio_xgb'], prob02), te)

26 bancarias nuevas -> 229 variables
lgbm amex + 26 bancarias                   meseta  423.4 | vs ref   -9.4 | máx  429.0 en 14000
ens3 con bancarias                         meseta  435.0 | vs ref   +2.2 | máx  441.1 en 14000
ens4 con bancarias + verscla02             meseta  433.5 | vs ref   +0.7 | máx  437.2 en 13500


433.4916666666666

## Fase 2 — Objetivo de ranking

En lugar de estimar una probabilidad, el modelo aprende a **ordenar** clientes (lo único que usamos: los primeros k).
Etiquetas graduadas: BAJA+2 = 2, BAJA+1 = 1, CONTINUA = 0, así aprende además que BAJA+2 vale más que BAJA+1.
Los clientes se reparten al azar en grupos de 1.000 (una "consulta" por grupo), como se hace en la industria.

In [5]:
prob_rk = lab.semillerio_rank(tr, te, feats, params, n_iter, semillas, objetivo='rank_xendcg')
lab.evaluar('ranking xendcg', prob_rk, te)
lab.evaluar('ens3 + ranking', lab.ensamble(probs['prob_junio_v05'], probs['prob_junio_v06'],
                                           probs['prob_junio_xgb'], prob_rk), te)

prob_lr = lab.semillerio_rank(tr, te, feats, params, n_iter, semillas, objetivo='lambdarank')
lab.evaluar('ranking lambdarank', prob_lr, te)

ranking xendcg                             meseta  415.2 | vs ref  -17.6 | máx  430.1 en 11000
ens3 + ranking                             meseta  435.5 | vs ref   +2.7 | máx  439.4 en 13500
ranking lambdarank                         meseta  428.4 | vs ref   -4.4 | máx  431.8 en 13500


428.35833333333335

## Fase 3 — Selección de variables (null importance)

Se entrena con el target real y 5 veces con el target mezclado al azar. Una variable que no supera claramente
a su importancia "por azar" solo agrega ruido. Probamos quedarnos con las que la superan.

In [6]:
ni = lab.null_importance(tr, feats, params, n_iter, semillas[0], n_null=5)
print(ni.head(15).round(1))
for umbral in [1.5, 3]:
    sel = ni.index[ni['cociente'] > umbral].tolist()
    p = lab.semillerio_lgb(tr, te, sel, params, n_iter, semillas)
    lab.evaluar(f'null importance > {umbral} ({len(sel)} vars)', p, te)

                             real  nula_max  nula_media  cociente
ctarjeta_visa               937.1       0.0         0.0     938.1
Visa_status                3719.3       9.6         1.9     351.1
Master_status              2193.4       5.9         1.2     318.0
ctarjeta_master            3547.3      10.3         3.1     314.7
pocos_productos            1164.8      21.7         5.6      51.4
cayo_actividad               25.0       0.0         0.0      26.0
cproductos_delta1          4148.6     175.8       135.3      23.5
ctrx_quarter              68064.5    3386.3      2913.2      20.1
bajo_productos              477.6      25.1        11.8      18.3
cdescubierto_preacordado   1338.7      72.4        32.1      18.3
ccajas_consultas           2610.5     149.7        76.2      17.3
cobra_sueldo               3058.7     198.6       167.8      15.3
cpayroll_trx              14406.6    1096.9       920.8      13.1
ccaja_seguridad            2314.7     192.3       134.3      12.0
ctrx_quart

## Registro de todo lo probado

In [7]:
lab.registro()

,fecha,experimento,meseta,vs_ref,max,k_max
0,2026-10-11 20:49,ref verscla05 (lgbm amex),426.9,-5.9,431.2,13000
1,2026-10-11 20:49,ref verscla07 (ens3),432.8,-0.0,436.7,13000
2,2026-10-11 20:51,"verscla02 (agente, abril->junio)",380.9,-51.9,396.6,15500
3,2026-10-11 20:51,ens2 verscla05 + verscla02,419.6,-13.2,422.9,13500
4,2026-10-11 20:51,ens4 ens3 + verscla02,433.1,0.3,437.8,14000
5,2026-10-11 20:52,lgbm amex + 26 bancarias,423.4,-9.4,429.0,14000
6,2026-10-11 20:52,ens3 con bancarias,435.0,2.2,441.1,14000
7,2026-10-11 20:52,ens4 con bancarias + verscla02,433.5,0.7,437.2,13500
8,2026-10-11 20:54,ranking xendcg,415.2,-17.6,430.1,11000
9,2026-10-11 20:54,ens3 + ranking,435.5,2.7,439.4,13500


## Entrega (plantilla)

Se completa con la combinación ganadora. Cada componente se reentrena con **junio** y predice **agosto**.
Las probabilidades de agosto de verscla05, verscla06 y XGBoost ya están guardadas; las de verscla02 vienen del pipeline.

In [8]:
# Ejemplo: ens4 = ens3 + verscla02
pf02 = pd.read_csv(ruta02 + 'prediccion_future.csv')[['numero_de_cliente', 'prob']]
prob_agosto_02 = lab.mismo_orden(pf02, fu)['prob'].values
assert not np.isnan(prob_agosto_02).any()

ens_agosto = lab.ensamble(probs['prob_agosto_v05'], probs['prob_agosto_v06'], probs['prob_agosto_xgb'], prob_agosto_02)
archivo = lab.entrega(ens_agosto, fu, 13500, 'verscla08_ens4')
print('coincide con verscla07:', lab.coincidencia(archivo, 'entregas/verscla07_ens3_k13500.csv'), '%')

entregas/verscla08_ens4_k13500.csv -> 13500 ids | 38799788 44781626 42520550 60622229 5715
coincide con verscla07: 96.3 %


In [9]:
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata

def oof_junio(df, feats, prm, n_it, semillas_cv, n_folds=5):
    """Predicción 'fuera de fold' para cada cliente de junio: entrena con el 80%, predice el 20%."""
    y = df['clase_binaria2'].values
    oof = np.zeros(len(df))
    for seed in semillas_cv:
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
        for tr_i, va_i in skf.split(df, df['clase_ternaria']):
            m = lgb.train({**prm, 'seed': seed, 'verbose': -1},
                          lgb.Dataset(df.iloc[tr_i][feats], label=y[tr_i]), num_boost_round=n_it)
            oof[va_i] += m.predict(df.iloc[va_i][feats])
    return oof / len(semillas_cv)

sem_cv = semillas[:3]

# Historia de 2 y 3 meses del agente, para junio (en abril estaba vacía)
hist = [c for c in fe02.columns if any(s in c for s in ['_lag2', '_lag3', '_delta2', '_delta3', '_slope3'])
        and c not in te.columns]
te_h = te.merge(fe02[fe02['foto_mes'] == 202106][['numero_de_cliente'] + hist], on='numero_de_cliente', how='left')
print(len(hist), 'variables de historia 2-3 meses | junio:', te_h.shape)

cv_A = oof_junio(te, feats, params, n_iter, sem_cv)
lab.evaluar('CV junio | A: verscla05 (203)', cv_A, te, referencia=0)

cv_B = oof_junio(te_h, feats + hist, params, n_iter, sem_cv)
lab.evaluar(f'CV junio | B: + historia 2-3 meses ({len(feats) + len(hist)})', cv_B, te, referencia=0)

cv_C = oof_junio(te02, feats02, params02, iter02, sem_cv)
lab.evaluar('CV junio | C: verscla02 (1256)', cv_C, te, referencia=0)

lab.evaluar('CV junio | A + C', lab.ensamble(cv_A, cv_C), te, referencia=0)
lab.evaluar('CV junio | B + C', lab.ensamble(cv_B, cv_C), te, referencia=0)
print(lab.gana_en_cortes(cv_B, cv_A, te))

708 variables de historia 2-3 meses | junio: (164114, 922)
CV junio | A: verscla05 (203)              meseta  494.2 | vs ref +494.2 | máx  504.4 en 12500


KeyboardInterrupt: 

In [ ]:
import sys
!{sys.executable} pipeline_v8.py --solo-final --semillas-final 10 --k 13000

[  0.0 min] params v05 {'objective': 'binary', 'boosting_type': 'gbdt', 'first_metric_only': True, 'boost_from_average': True, 'feature_pre_filter': False, 'max_bin': 31, 'bagging_freq': 1, 'verbose': -1, 'num_leaves': 76, 'learning_rate': 0.009414528604172152, 'min_data_in_leaf': 691, 'feature_fraction': 0.8075276247459833, 'bagging_fraction': 0.5141471765041344} | árboles 655
[  0.1 min] 152 variables numéricas en el crudo
[  0.2 min] features listas
[  0.2 min] 484 variables en el modelo
[  0.2 min] MODELO FINAL: 202103-202106 -> 202108
[  0.3 min]   entreno 10 semillas | 654,066 filas | 484 variables | min_data_in_leaf 691 -> 2768
[  1.0 min]     semilla 1/10 (100129) lista
[  1.7 min]     semilla 2/10 (100183) lista
[  2.4 min]     semilla 3/10 (100151) lista


In [ ]:
!{sys.executable} pipeline_v8.py --solo-final --semillas-final 10 --k 13000